<a href="https://colab.research.google.com/github/omarsamehabobaker619-bot/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarsamehabobaker619-bot/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Logistic Regression**

My label (is_underperforming) is a yes/no target — this week's session specifically
recommends starting with the smallest method for this shape of question:
Rule → Logistic → Tree → Forest. I already have the Rule (Week 4 baseline), so
Logistic Regression is the natural next step: it's still readable (each feature
gets a coefficient showing direction and rough size of its effect), and it lets me
test honestly whether a real learned model beats my hand-written rule before
reaching for anything more complex like Random Forest.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub", "pandas", "scikit-learn"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import duckdb, pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

# SAME filters as Week 4 baseline — frozen, not to be changed after seeing results
df = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        AVG(gsc_avg_position) AS avg_position,
        LN(SUM(gsc_impressions) + 1) AS log_impressions,
        SUM(ga4_engaged_sessions) AS engaged_sessions,
        SUM(ga4_total_engagement_sec) AS total_engagement_sec,
        SUM(scroll_events) AS scroll_events,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_impressions) AS total_impressions
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
    ORDER BY content_hash_id
""").df()

df["ctr"] = df["total_clicks"] / df["total_impressions"]
df["position_bucket"] = pd.cut(df["avg_position"], bins=[0, 3, 10, 20, 50, 1000], labels=["1-3","4-10","11-20","21-50","50+"])
median_ctr_by_bucket = df.groupby("position_bucket", observed=True)["ctr"].transform("median")
df["is_underperforming"] = (df["ctr"] <= median_ctr_by_bucket).astype(int)

# Rebuild Week 4 baseline score for later comparison — frozen, same logic as before
in_target_volume_band = (df["total_impressions"] >= 100) & (df["total_impressions"] <= 5000)
df["baseline_score"] = df["is_underperforming"] * in_target_volume_band.astype(int) * df["total_impressions"]

print("Rows:", len(df))
print("Unique clients:", df["client_hash_id"].nunique())
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 176738
Unique clients: 47


,client_hash_id,content_hash_id,avg_position,log_impressions,engaged_sessions,total_engagement_sec,scroll_events,total_clicks,total_impressions,ctr,position_bucket,is_underperforming,baseline_score
0,client_9958f0a7ae1df715,content_000005d4ced12088,72.854861,4.465908,0.0,0.0,0.0,0.0,86.0,0.000000,50+,1,0.0
1,client_73cda7b4e4f265ea,content_00007bd2985b77c3,5.269565,3.871201,0.0,0.0,0.0,0.0,47.0,0.000000,4-10,1,0.0
2,client_3ffa76342f366962,content_0000cd28fbda69f3,4.251282,3.401197,0.0,0.0,0.0,0.0,29.0,0.000000,4-10,1,0.0
3,client_2094c6eb080311d5,content_0000d495bfbfb4a8,3.333333,2.772589,0.0,0.0,0.0,0.0,15.0,0.000000,4-10,1,0.0
4,client_08a6a72ff48e62c0,content_00014efc121d911d,4.964683,4.762174,NaN,NaN,NaN,1.0,116.0,0.008621,4-10,0,0.0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split: grouped by client, 70/30**

I split by client_hash_id, not by random row, using a 70/30 train/test split
(GroupShuffleSplit, random_state=42 for reproducibility). This guarantees no
client's pages appear in both train and test — confirmed by checking client
overlap = 0.

This matters because pages from the same client likely share patterns (site
structure, content style, industry) that have nothing to do with the general
relationship between position/engagement and CTR underperformance. A random
row-level split could let the model quietly learn "this looks like Client X's
pages" rather than the real signal, inflating its test score without teaching
it anything that generalizes to a new client it's never seen — which is the
actual real-world use case for this rule.

Result: 130,904 rows / 32 clients in train, 45,834 rows / 15 clients in test.

In [2]:
from sklearn.model_selection import GroupShuffleSplit

# Grouped split by client — no client's pages appear in both train and test,
# preventing the model from learning client-specific quirks instead of
# general patterns (same leakage principle as Week 3, applied to splitting)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(splitter.split(df, groups=df["client_hash_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

print("Train rows:", len(train_df), "| Train clients:", train_df["client_hash_id"].nunique())
print("Test rows:", len(test_df), "| Test clients:", test_df["client_hash_id"].nunique())

# Confirm no overlap in clients between train and test
overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print("Client overlap between train/test:", len(overlap), "(should be 0)")

Train rows: 130904 | Train clients: 32
Test rows: 45834 | Test clients: 15
Client overlap between train/test: 0 (should be 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

**Model vs. baseline: baseline wins, narrowly**

At Precision@50, the Week 4 rule-based baseline scored 1.00, while Logistic
Regression scored 0.98 — both computed on the same test set (45,834 rows, 15
held-out clients), same split, same metric. The base rate is 0.557, so both
methods are performing well above chance.

The baseline slightly outperforming the model makes sense here: my rule's
underlying signal (CTR compared to the position-bucket median) is essentially
the same information used to build the label itself, so a simple, transparent
rule already captures nearly all of it. This is a genuine, useful finding — it
means, for this specific decision, added model complexity doesn't currently earn
its place over the honest baseline. Per this week's rule, I'm reporting this as-is
rather than adjusting filters or metrics to make the model look better.

In [3]:
from sklearn.linear_model import LogisticRegression
import numpy as np

honest_features = ["avg_position", "log_impressions", "engaged_sessions", "total_engagement_sec", "scroll_events"]

X_train = train_df[honest_features].fillna(0)
y_train = train_df["is_underperforming"]
X_test = test_df[honest_features].fillna(0)
y_test = test_df["is_underperforming"]

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)

model_probs = model.predict_proba(X_test)[:, 1]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

k = 50
model_p50 = precision_at_k(model_probs, y_test.values, k)
baseline_p50 = precision_at_k(test_df["baseline_score"].values, y_test.values, k)
base_rate = y_test.mean()

comparison = pd.DataFrame({
    "method": ["Baseline (Week 4 rule)", "Logistic Regression"],
    f"precision@{k}": [baseline_p50, model_p50],
    "base_rate": [base_rate, base_rate]
})

print(comparison)

                   method  precision@50  base_rate
0  Baseline (Week 4 rule)          1.00   0.556508
1     Logistic Regression          0.98   0.556508


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# What does the model lean on? (coefficients show direction + rough size)
coef_df = pd.DataFrame({
    "feature": honest_features,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)

print("Feature coefficients (sorted by size):")
print(coef_df)

# Where is the model most wrong? Look at the top 50 it predicted,
# and find any that were actually NOT underperforming (false positives)
test_df_copy = test_df.copy()
test_df_copy["model_prob"] = model_probs
test_df_copy["true_label"] = y_test.values

top50 = test_df_copy.sort_values("model_prob", ascending=False).head(50)
wrong_in_top50 = top50[top50["true_label"] == 0]

print("\nWrong predictions in top 50:", len(wrong_in_top50))
print(wrong_in_top50[["content_hash_id", "avg_position", "ctr", "model_prob", "true_label"]].head(5))

Feature coefficients (sorted by size):
                feature  coefficient
2      engaged_sessions    -1.636017
1       log_impressions    -0.782015
0          avg_position     0.047075
4         scroll_events    -0.001135
3  total_engagement_sec    -0.000331

Wrong predictions in top 50: 1
                content_hash_id  avg_position  ctr  model_prob  true_label
24109  content_8717cb37cf1fb1bc         257.0  0.5         1.0           0


**Where the model is wrong**

Only 1 of the top 50 predictions was wrong — content_8717cb37cf1fb1bc, at a very
poor position (257) but with an unusually high CTR (0.5). The model predicted it
was underperforming (prob=1.0) based on its poor position, but its true label says
otherwise. This is most likely a small-sample artifact: a page ranked this poorly
probably received very few impressions, so a couple of lucky clicks produced a
misleadingly high CTR that isn't a stable, real pattern.

**What the model leans on**

engaged_sessions (-1.64) and log_impressions (-0.78) are by far the strongest
signals, both pushing toward "not underperforming" as engagement and traffic
increase — a sensible relationship, since genuinely engaged visitors are more
likely to have also clicked through. avg_position has a small positive effect
(0.047) — worse position slightly raises the underperformance prediction, but
it's a much weaker signal than engagement here. scroll_events and
total_engagement_sec barely matter (near-zero coefficients).

None of these coefficients look suspiciously dominant or perfect — the signal is
spread across several features rather than concentrated in one, which is a good
sign against hidden leakage (unlike the deliberate total_clicks trap from Week 3,
which produced one overwhelming, unrealistic signal).

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all) — confirm
  this now before checking this box
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
  (e.g. "the baseline slightly outperforming the model" rather than overclaiming;
  "most likely a small-sample artifact" rather than asserting certainty)
- [x] Committed to my repo under work/notebooks/